# Study 947 — The Buffer Ladder 🪜

**Does laddering a buffer fund add anything you could not do yourself with four trades?**

A buffer ETF's terms — the stated buffer against losses, the stated cap on gains — are
struck once a year on a named month. So the payoff you actually get depends on *when you
bought*. The industry's answer is the **laddered wrapper**: one ticker (**BUFR**) holding a
spread of vintages so entry timing averages out, for a management fee **on top of** the
underlying funds' expense ratios.

The obvious retort: buy the vintages yourself and equal-weight them. Four trades.

We race **BUFR** against each of its four quarterly vintages (**PJAN / PAPR / PJUL /
POCT**), against an equal-weight **DIY basket** of them, against a **beta-matched** DIY
ladder, and against the dumb **SPY/BIL** mix — all **excess-of-cash**, all total-return,
2020-08-11 → 2026-06-30 (1,477 days), 5 bps one-way.

*Every real number below is the frozen headline from `docs/results.md` (Fingerprint
`489cd6cd95e2`); the live cells run offline synthetic demonstrations and say so. As-of
2026-06-30.*


## 1. The problem laddering claims to solve

PJAN resets every January, PAPR every April, and so on. Buy one of them in the middle of its outcome period and you inherit whatever the market has already done to that vintage's buffer and cap. Pick the wrong door and you get a different year from your neighbour who picked the right one.

So: how big is that difference, really? Here is the gap between the best-performing and worst-performing vintage over every rolling one-year window on the tape.

| | Best minus worst vintage, rolling 1 year |
|---|--:|
| Mean | **4.53 pp** |
| Median | 4.51 pp |
| Worst case | 8.70 pp |

About four and a half percentage points, typically. Not nothing — but hold that number, because the next section is where the story turns.

## 2. The catch — the four vintages are almost the same fund

The four vintages have a **0.889** daily correlation with each other. They all track the same index with a similar damping; the reset month shifts the strikes a little, but the underlying is identical.

That matters enormously, because averaging things that move together removes almost no risk. Averaging things that move independently removes a lot. Here is the arithmetic, and it is not a market fact — it is a fact about averages:

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))
import numpy as np
from buffer_ladder import data, strategy as st

In [2]:
# SYNTHETIC — no market data. How much variance does averaging N legs remove,
# as a function of how correlated those legs are? Closed form: for N equally
# correlated legs of equal variance, sd(basket)/sd(leg) = sqrt((1 + (N-1)*rho) / N).
def variance_reduction(rho, n=4):
    return (1.0 - np.sqrt((1.0 + (n - 1) * rho) / n)) * 100.0

for rho in (0.0, 0.25, 0.50, 0.75, 0.889, 0.95):
    tag = '   <-- the four Power Buffer vintages' if rho == 0.889 else ''
    print('correlation %.3f  ->  averaging 4 legs cuts sd by %5.1f%%%s'
          % (rho, variance_reduction(rho), tag))

correlation 0.000  ->  averaging 4 legs cuts sd by  50.0%
correlation 0.250  ->  averaging 4 legs cuts sd by  33.9%
correlation 0.500  ->  averaging 4 legs cuts sd by  20.9%
correlation 0.750  ->  averaging 4 legs cuts sd by   9.9%
correlation 0.889  ->  averaging 4 legs cuts sd by   4.3%   <-- the four Power Buffer vintages
correlation 0.950  ->  averaging 4 legs cuts sd by   1.9%


## 3. What that means on the real tape

On the real tape the DIY basket's one-year return standard deviation is **6.61%** against **6.77%** for the average single vintage — a **2.4%** reduction. Roughly a sixth of a percentage point.

That is the size of the prize laddering is competing for. And it is free: four trades and a rebalance reminder get you all of it.

> 🔬 **For the quants** — the cell above is the equally-correlated-legs closed form sd(basket)/sd(leg) = √((1 + (N−1)ρ)/N). At ρ = 0.889 and N = 4 it predicts a **4.2%** cut in daily standard deviation; the tape delivers **4.20%** — the arithmetic is exact. On a one-year *holding period*, where each vintage's path through its own buffer and cap matters, the realised cut is smaller still at **2.4%**. Either way there is nothing left over for the wrapper to have been clever about.

## 4. So did the wrapper win anyway?

On raw return, yes — and this is where it gets interesting.

In [3]:
R = dict(bufr=(7.93, 10.21, 0.777, 2.06), diy=(6.61, 7.74, 0.853, 2.3), spy=(13.73, 16.92, 0.812, 2.17), gap_diy=(1.33, 1.18, 3.52, -0.076),
         beta_bufr=0.579, beta_diy=0.439, gap_matched=(-0.96, -1.69, 2.35, -0.057))
print('BUFR (the ladder) : %+.2f%% a year, vol %.2f%%, excess Sharpe %+.3f'
      % R['bufr'][:3])
print('DIY basket        : %+.2f%% a year, vol %.2f%%, excess Sharpe %+.3f'
      % R['diy'][:3])
print()
print('the wrapper earned %+.2f pp/yr more ... and its Sharpe is %+.3f LOWER'
      % (R['gap_diy'][0], R['gap_diy'][3]))

BUFR (the ladder) : +7.93% a year, vol 10.21%, excess Sharpe +0.777
DIY basket        : +6.61% a year, vol 7.74%, excess Sharpe +0.853

the wrapper earned +1.33 pp/yr more ... and its Sharpe is -0.076 LOWER


## 5. Because the wrapper is not a better ladder — it is a bolder one

BUFR's sensitivity to the S&P 500 is **0.579**. The four-vintage DIY basket's is **0.439**. The wrapper simply holds more equity exposure — and in a five-year stretch where the S&P returned +13.7% a year excess of cash, more equity exposure means more return.

That is not a laddering premium. It is a beta you can buy for the price of an index fund. Top the DIY basket up with SPY until it carries the *same* beta as the wrapper, and the wrapper **loses** by **-0.96 pp/yr** (HAC *t* = -1.69) — about the size of the extra fee layer it charges.

The tell is in the risk numbers: the wrapper's worst loss on the tape was **-13.7%**, against **-10.8%** for the home-made basket and **-10.2%** for POCT held alone. The product you bought *for downside protection* protected you least.

## 6. The single year that mattered

There has been exactly one genuine down-year in BUFR's life. Here is how each arm did in it, alongside the rest of the tape (nominal total return, %):

| Year | BUFR | PJAN | PAPR | PJUL | POCT | DIY basket | SPY |
|---|--:|--:|--:|--:|--:|--:|--:|
| 2021 | **+11.88** | +8.80 | +7.51 | +7.20 | +9.45 | **+8.25** | +28.73 |
| 2022 | **-7.57** | -5.29 | -4.29 | -2.08 | -1.25 | **-3.20** | -18.18 |
| 2023 | **+19.63** | +18.18 | +16.45 | +19.87 | +20.12 | **+18.66** | +26.18 |
| 2024 | **+14.68** | +13.45 | +12.28 | +13.76 | +9.55 | **+12.26** | +24.89 |
| 2025 | **+12.44** | +11.29 | +6.58 | +12.78 | +10.99 | **+10.40** | +17.72 |

BUFR out-returned the DIY basket every complete year — and in **2022**, the only year anyone actually needed a buffer, it lost **more than double** what the home-made basket lost, and more than every single one of its own constituents. That one row is the whole beta story in miniature.

## 7. Is any of this statistically real? (No.)

That is the honest headline. **Not one comparison on this tape clears the desk's |*t*| = 2 bar, in either direction.**

| | Gap | HAC *t* | Bootstrap 95% CI |
|---|--:|--:|--:|
| Wrapper vs DIY basket | +1.33 pp/yr | +1.18 | [-0.81, +3.22] |
| Wrapper vs **beta-matched** DIY | -0.96 pp/yr | -1.69 | [-1.91, -0.05] |

The beta-matched line is the one to be careful with: its bootstrap CI does clear zero at the 21-day block used above, but flip to a 5- or 10-day block and it straddles zero again (notebook 02, §5). The exclusion is the block length talking, so the HAC *t* of −1.69 is what we stamp on.

Five-point-nine years and one down-year is simply not enough tape to resolve a one-percentage-point effect. What we *can* say without a *t*-test is the mechanism: the vintages are nearly the same fund, so laddering them removes nearly nothing, and the wrapper's visible edge is beta.

## 8. Live check — the machinery is not broken (offline synthetic)

Before believing a null, check the detector can find something. On a synthetic panel with a *planted* laddering premium it recovers it cleanly; on a null panel it stays quiet. Nothing below touches market data.

In [4]:
# SYNTHETIC control — the machinery proof. Never supports the real-tape stamp.
for tag, ss, fee in [('planted premium', 1.0, 0.002),
                     ('null, fee only ', 0.0, 0.002),
                     ('null, no fee   ', 0.0, 0.000)]:
    px, truth = data.synthetic_panel(signal_strength=ss, extra_fee_ann=fee, seed=947)
    d = st.synthetic_detect(px, truth)
    print('%s : planted %+5.2f pp/yr -> recovered %+5.2f (error %+.2f), HAC t %+.2f'
          % (tag, d['expected_gap_pp'], d['gap_ann_pp'], d['error_pp'], d['t_hac']))

nulls = [st.synthetic_detect(*data.synthetic_panel(signal_strength=0.0,
                                                   extra_fee_ann=0.0, seed=947 + s))
         for s in range(8)]
ts = np.array([n['t_hac'] for n in nulls])
gaps = np.array([n['gap_ann_pp'] for n in nulls])
print('\nnull across 8 seeds: mean gap %+.2f pp/yr (sd %.2f), max |t| %.2f, fires on %d/8'
      % (gaps.mean(), gaps.std(ddof=1), np.abs(ts).max(), int((np.abs(ts) >= 2).sum())))

planted premium : planted +3.80 pp/yr -> recovered +4.16 (error +0.36), HAC t +4.84


null, fee only  : planted -0.20 pp/yr -> recovered +0.16 (error +0.36), HAC t +0.18


null, no fee    : planted +0.00 pp/yr -> recovered +0.36 (error +0.36), HAC t +0.42



null across 8 seeds: mean gap +0.11 pp/yr (sd 0.87), max |t| 1.69, fires on 0/8


## Verdict

- **Signal — None.** No comparison clears |*t*| = 2: +1.33 pp/yr vs the DIY basket (*t* = +1.18) and -0.96 pp/yr vs a beta-matched DIY ladder (*t* = -1.69). The wrapper's apparent edge is **beta** (0.579 vs 0.439), and on risk-adjusted terms it is *worse* than the basket it wraps (+0.777 vs +0.853) with a deeper drawdown. The entry-point luck it exists to average away is worth a **2.4%** variance reduction, because the vintages are 0.889 correlated.
- **Tradability — Mirage.** Nothing to bank in either direction. And one caveat that matters more than the *t*-stats: there is exactly **one** laddered wrapper with this history, over 5.9 years containing a single down-year. This is an n-of-1 product test, not a cross-section.
- **What the ladder is genuinely good for:** one trade instead of four, no rebalance calendar, no vintage to choose. That convenience is real, and this study does not price it. It is just not an edge.